# Planetary-nebula Central Stars as Tracers of Planetary-nebula–Star-cluster Associations in the Galaxy

Paper: https://iopscience.iop.org/article/10.3847/1538-4357/ae9f58

---

# Part 1. Evolution of the Central Star of a Planetary Nebula

In [ ]:
# Planetary Nebula / Post-AGB evolution
# Single Jupyter cell
#
# Output:
#   planetary_nebula_evolution.mp4
#   planetary_nebula_evolution.webm
#
# Requires:
#   numpy, matplotlib, ffmpeg
#
# IMPORTANT:
# The evolutionary track is SCHEMATIC.
# It illustrates the qualitative post-AGB evolution described in the paper,
# not digitized Miller Bertolami (2016) track data.

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Circle
from matplotlib.collections import LineCollection
from IPython.display import Video, display
import shutil
import os

# ============================================================
# SETTINGS
# ============================================================

OUTPUT = "webm"          # "webm" or "mp4"
FPS = 30
DURATION = 16            # seconds
DPI = 120

W, H = 12.8, 7.2        # 16:9 figure

if OUTPUT.lower() == "webm":
    OUTFILE = "planetary_nebula_evolution.webm"
else:
    OUTFILE = "planetary_nebula_evolution.mp4"

NFRAMES = FPS * DURATION

if shutil.which("ffmpeg") is None:
    raise RuntimeError(
        "FFmpeg not found. Install ffmpeg and make sure it is available in PATH."
    )

# Reproducible stars
rng = np.random.default_rng(42)

# ============================================================
# SCHEMATIC POST-AGB TRACK
# ============================================================
#
# HR-like diagram:
# x = log10(Teff / K)
# y = log10(L / Lsun)
#
# Evolution:
# 1. heating at nearly constant luminosity
# 2. turn toward WD cooling track
# 3. cooling/fading

n1, n2, n3 = 180, 90, 210

# Heating phase
x1 = np.linspace(3.72, 5.08, n1)
y1 = 3.72 - 0.05 * ((x1 - 3.72) / (5.08 - 3.72))**1.5

# Turn / luminosity decline
u = np.linspace(0, 1, n2)
x2 = 5.08 + 0.06 * np.sin(np.pi * u)
y2 = 3.67 - 1.10 * u**1.25

# White dwarf cooling
u = np.linspace(0, 1, n3)
x3 = 5.08 - 0.63 * u
y3 = 2.57 - 2.25 * u**1.15

track_x = np.concatenate([x1, x2, x3])
track_y = np.concatenate([y1, y2, y3])

# Resample to animation frame count
src = np.linspace(0, 1, len(track_x))
dst = np.linspace(0, 1, NFRAMES)

evol_x = np.interp(dst, src, track_x)
evol_y = np.interp(dst, src, track_y)

# ============================================================
# FIGURE
# ============================================================

fig = plt.figure(figsize=(W, H), facecolor="#03060d")

# Left: HR diagram
ax = fig.add_axes([0.07, 0.13, 0.53, 0.76])
ax.set_facecolor("#050914")

# Right: PN visualization
pn_ax = fig.add_axes([0.63, 0.13, 0.33, 0.76])
pn_ax.set_facecolor("#03060d")
pn_ax.set_xlim(-1.15, 1.15)
pn_ax.set_ylim(-1.15, 1.15)
pn_ax.set_aspect("equal")
pn_ax.axis("off")

# ============================================================
# BACKGROUND STAR FIELD
# ============================================================

sx = rng.uniform(-1.1, 1.1, 160)
sy = rng.uniform(-1.1, 1.1, 160)
ss = rng.uniform(0.4, 5, 160)
sa = rng.uniform(0.08, 0.45, 160)

pn_ax.scatter(
    sx, sy,
    s=ss,
    c="white",
    alpha=sa,
    linewidths=0,
    zorder=0
)

# ============================================================
# HR DIAGRAM
# ============================================================

ax.set_xlim(3.55, 5.35)
ax.set_ylim(0.0, 4.25)

ax.set_xlabel(
    r"$\log\,T_{\rm eff}\ [{\rm K}]$",
    color="#cbd6e2",
    fontsize=11
)
ax.set_ylabel(
    r"$\log\,L/L_\odot$",
    color="#cbd6e2",
    fontsize=11
)

ax.tick_params(colors="#8293a6", labelsize=9)

for spine in ax.spines.values():
    spine.set_color("#24364a")

ax.grid(
    True,
    color="#203044",
    linewidth=0.6,
    alpha=0.35
)

# Hotter stars traditionally toward left in HR diagrams
ax.invert_xaxis()

# Full track as subtle background
ax.plot(
    track_x,
    track_y,
    color="#38516c",
    lw=1.3,
    alpha=0.5
)

# Track label
ax.text(
    0.03, 0.965,
    "SCHEMATIC POST-AGB EVOLUTION",
    transform=ax.transAxes,
    color="#7894ad",
    fontsize=8,
    va="top"
)

# Animated travelled path
trail, = ax.plot(
    [],
    [],
    color="#75cfff",
    lw=2.2,
    alpha=0.9
)

# Glow layers for moving star
star_glow_big, = ax.plot(
    [],
    [],
    "o",
    ms=18,
    color="#68caff",
    alpha=0.08
)

star_glow, = ax.plot(
    [],
    [],
    "o",
    ms=10,
    color="#9edfff",
    alpha=0.25
)

star_point, = ax.plot(
    [],
    [],
    "o",
    ms=4.5,
    color="white"
)

# ============================================================
# PN OBJECT
# ============================================================

# Multiple shells give a cheap glow effect without image filters
shells = []

for radius, lw, alpha in [
    (0.35, 18, 0.025),
    (0.35, 11, 0.045),
    (0.35,  6, 0.075),
    (0.35,  2, 0.20),
]:
    c = Circle(
        (0, 0),
        radius,
        fill=False,
        edgecolor="#67d9ff",
        linewidth=lw,
        alpha=alpha
    )
    pn_ax.add_patch(c)
    shells.append(c)

# Inner warmer shell
inner_shell = Circle(
    (0, 0),
    0.22,
    fill=False,
    edgecolor="#b38cff",
    linewidth=2.0,
    alpha=0.25
)
pn_ax.add_patch(inner_shell)

# Central star glow
central_glows = []

for size, alpha in [
    (650, 0.025),
    (300, 0.05),
    (120, 0.12),
    (35, 0.95)
]:
    sc = pn_ax.scatter(
        [0], [0],
        s=size,
        c="white",
        alpha=alpha,
        linewidths=0,
        zorder=10
    )
    central_glows.append(sc)

# A few faint radial filaments
angles = rng.uniform(0, 2*np.pi, 42)
base_lengths = rng.uniform(0.27, 0.47, len(angles))

segments = []

for a, length in zip(angles, base_lengths):
    r0 = length * rng.uniform(0.65, 0.85)
    r1 = length
    segments.append([
        [np.cos(a)*r0, np.sin(a)*r0],
        [np.cos(a)*r1, np.sin(a)*r1]
    ])

filaments = LineCollection(
    segments,
    colors="#79dfff",
    linewidths=0.6,
    alpha=0.12
)
pn_ax.add_collection(filaments)

# ============================================================
# TEXT
# ============================================================

fig.text(
    0.07, 0.94,
    "PLANETARY NEBULA — CENTRAL STAR EVOLUTION",
    color="white",
    fontsize=16,
    weight="bold"
)

fig.text(
    0.07, 0.905,
    "From envelope ejection to the white-dwarf cooling track",
    color="#7f96ad",
    fontsize=9
)

phase_text = pn_ax.text(
    0.5, 0.08,
    "",
    transform=pn_ax.transAxes,
    ha="center",
    color="white",
    fontsize=10,
    weight="bold"
)

data_text = ax.text(
    0.03, 0.06,
    "",
    transform=ax.transAxes,
    color="#9db0c2",
    fontsize=9,
    family="monospace"
)

# ============================================================
# ANIMATION
# ============================================================

def smoothstep(x):
    x = np.clip(x, 0, 1)
    return x*x*(3 - 2*x)

def update(frame):

    f = frame / (NFRAMES - 1)

    x = evol_x[frame]
    y = evol_y[frame]

    # --------------------------------------------------------
    # HR diagram
    # --------------------------------------------------------

    trail.set_data(evol_x[:frame+1], evol_y[:frame+1])

    star_glow_big.set_data([x], [y])
    star_glow.set_data([x], [y])
    star_point.set_data([x], [y])

    Teff = 10**x
    Lum = 10**y

    data_text.set_text(
        f"log Teff = {x:4.2f}\n"
        f"log L/L☉ = {y:4.2f}\n"
        f"Teff ≈ {Teff:,.0f} K"
    )

    # --------------------------------------------------------
    # Evolutionary phases
    # --------------------------------------------------------

    if f < 0.12:
        phase = "ENVELOPE EJECTION"
    elif f < 0.52:
        phase = "PLANETARY NEBULA"
    elif f < 0.68:
        phase = "HOT CENTRAL STAR"
    else:
        phase = "WHITE-DWARF COOLING"

    phase_text.set_text(phase)

    # --------------------------------------------------------
    # Nebula expansion / fading
    # --------------------------------------------------------

    # Formation
    formation = smoothstep((f - 0.03) / 0.15)

    # Expansion throughout PN stage
    expansion = smoothstep((f - 0.06) / 0.72)

    # Fade after central star enters cooling track
    fade = 1.0 - 0.92 * smoothstep((f - 0.60) / 0.36)

    nebula_alpha = formation * fade

    radius = 0.10 + 0.73 * expansion

    alpha_factors = [0.10, 0.18, 0.32, 0.75]

    for shell, af in zip(shells, alpha_factors):
        shell.set_radius(radius)
        shell.set_alpha(nebula_alpha * af)

    inner_shell.set_radius(radius * 0.61)
    inner_shell.set_alpha(nebula_alpha * 0.45)

    # --------------------------------------------------------
    # Filament expansion
    # --------------------------------------------------------

    new_segments = []

    for a, length in zip(angles, base_lengths):

        scale = 0.25 + 1.55 * expansion

        r0 = length * scale * 0.72
        r1 = length * scale

        new_segments.append([
            [np.cos(a)*r0, np.sin(a)*r0],
            [np.cos(a)*r1, np.sin(a)*r1]
        ])

    filaments.set_segments(new_segments)
    filaments.set_alpha(nebula_alpha * 0.32)

    # --------------------------------------------------------
    # Central star brightness
    # --------------------------------------------------------

    # Strong during hot-CS stage, fades on WD cooling branch
    brightness = 0.35 + 0.65 * (1.0 - smoothstep((f - 0.68) / 0.30))

    glow_alphas = [
        0.035 * brightness,
        0.08 * brightness,
        0.20 * brightness,
        0.95
    ]

    for obj, alpha in zip(central_glows, glow_alphas):
        obj.set_alpha(alpha)

    return (
        trail,
        star_glow_big,
        star_glow,
        star_point,
        phase_text,
        data_text,
        inner_shell,
        filaments,
        *shells,
        *central_glows
    )

anim = FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    interval=1000/FPS,
    blit=False
)

# ============================================================
# EXPORT
# ============================================================

print(f"Rendering {NFRAMES} frames → {OUTFILE}")

if OUTPUT.lower() == "webm":

    writer = FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2200,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-row-mt", "1",
            "-deadline", "good",
            "-cpu-used", "2"
        ]
    )

else:

    writer = FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=3500,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-preset", "medium",
            "-movflags", "+faststart"
        ]
    )

anim.save(
    OUTFILE,
    writer=writer,
    dpi=DPI
)

plt.close(fig)

size_mb = os.path.getsize(OUTFILE) / (1024 * 1024)

print(f"Done: {OUTFILE}")
print(f"Size: {size_mb:.1f} MB")

display(
    Video(
        OUTFILE,
        embed=True,
        html_attributes="controls loop autoplay muted"
    )
)


# Эволюция центральной звезды планетарной туманности

Планетарная туманность возникает на поздней стадии эволюции звезды небольшой или средней массы. Покидая асимптотическую ветвь гигантов (AGB), звезда сбрасывает внешние слои, образующие расширяющуюся газовую оболочку. Оставшееся горячее ядро становится центральной звездой планетарной туманности.

На диаграмме показана схематическая эволюция центральной звезды. Сначала её эффективная температура быстро растёт, тогда как светимость остаётся примерно постоянной. Интенсивное ультрафиолетовое излучение горячего ядра ионизирует выброшенный газ, заставляя планетарную туманность светиться. Затем светимость центральной звезды начинает снижаться, и она переходит на ветвь охлаждения белых карликов. Одновременно оболочка продолжает расширяться и постепенно рассеивается в межзвёздном пространстве.

Именно положение центральных звёзд на таких эволюционных треках авторы исследования используют как один из способов проверки предполагаемой связи планетарных туманностей со звёздными скоплениями.  

Анимация носит иллюстративный характер. Показанный эволюционный трек является схематическим и не представляет собой оцифрованные данные моделей Miller Bertolami (2016).


# Evolution of the Central Star of a Planetary Nebula

A planetary nebula forms during a late stage in the evolution of a low- or intermediate-mass star. As the star leaves the asymptotic giant branch (AGB), it ejects its outer layers, producing an expanding shell of gas. The remaining hot core becomes the central star of the planetary nebula.

The diagram shows the central star’s evolution schematically. Its effective temperature initially rises rapidly while its luminosity remains approximately constant. Intense ultraviolet radiation from the hot stellar core ionizes the ejected gas, causing the planetary nebula to glow. The central star subsequently begins to lose luminosity and enters the white-dwarf cooling track, while the surrounding nebular shell continues to expand and gradually disperses into interstellar space.

In the study, the positions of central stars relative to such evolutionary tracks provide one of the tests used to assess proposed associations between planetary nebulae and Galactic star clusters.  

The animation is illustrative. The evolutionary track shown is schematic and does not represent digitized data from the Miller Bertolami (2016) models.

---

# Part 2. How to Identify a Planetary Nebula–Star Cluster Association

In [ ]:
# ============================================================
# FINDING A PLANETARY NEBULA–STAR CLUSTER ASSOCIATION
# ============================================================
#
# Single Jupyter cell
#
# Output:
#   pn_cluster_association.webm
#   or
#   pn_cluster_association.mp4
#
# Requires:
#   numpy
#   matplotlib
#   ffmpeg
#
# The animation schematically illustrates the selection
# procedure described in Fragkou et al. (2026).
#
# Numbers shown:
#   277 positional pairs
#    38 kinematic candidates
#    15 extinction-compatible candidates
#     7 higher-likelihood associations
#
# IMPORTANT:
# The star cluster and planetary nebula shown here are schematic.
# They do not represent a specific observed cluster.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Circle, FancyBboxPatch
from IPython.display import Video, display
import shutil
import os


# ============================================================
# SETTINGS
# ============================================================

OUTPUT = "webm"       # "webm" or "mp4"

FPS = 30
DURATION = 18         # seconds
DPI = 120

FIG_W = 12.8
FIG_H = 7.2

NFRAMES = FPS * DURATION

if OUTPUT.lower() == "webm":
    OUTFILE = "pn_cluster_association.webm"
else:
    OUTFILE = "pn_cluster_association.mp4"


if shutil.which("ffmpeg") is None:
    raise RuntimeError(
        "FFmpeg not found. Install ffmpeg and make sure "
        "it is available in PATH."
    )


# Reproducible random star field
rng = np.random.default_rng(19)


# ============================================================
# FIGURE
# ============================================================

fig = plt.figure(
    figsize=(FIG_W, FIG_H),
    facecolor="#03060d"
)

# Left: star-cluster field
ax = fig.add_axes([
    0.04,
    0.10,
    0.58,
    0.80
])

ax.set_facecolor("#03060d")
ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-1.0, 1.0)
ax.set_aspect("equal")
ax.axis("off")


# Right: selection panel
panel = fig.add_axes([
    0.65,
    0.10,
    0.31,
    0.80
])

panel.set_facecolor("#050914")
panel.set_xlim(0, 1)
panel.set_ylim(0, 1)
panel.axis("off")


# ============================================================
# TITLE
# ============================================================

fig.text(
    0.055,
    0.94,
    "FINDING A PLANETARY NEBULA–STAR CLUSTER ASSOCIATION",
    color="white",
    fontsize=15,
    weight="bold"
)

fig.text(
    0.055,
    0.905,
    "From positional coincidence to a physically plausible association",
    color="#8196aa",
    fontsize=9
)


# ============================================================
# STAR CLUSTER
# ============================================================

N_STARS = 270

# Centrally concentrated distribution
theta = rng.uniform(
    0,
    2 * np.pi,
    N_STARS
)

r = np.clip(
    rng.exponential(
        0.36,
        N_STARS
    ),
    0,
    1.05
)

xs = r * np.cos(theta)
ys = 0.78 * r * np.sin(theta)


# Different apparent stellar sizes
sizes = rng.lognormal(
    mean=0.25,
    sigma=0.75,
    size=N_STARS
)

sizes = np.clip(
    sizes,
    0.5,
    15
)


# Individual stellar brightness
alphas = rng.uniform(
    0.18,
    0.75,
    N_STARS
)


# ------------------------------------------------------------
# IMPORTANT FIX:
#
# Instead of passing alpha=array to scatter and later replacing
# it with a scalar alpha, we explicitly create RGBA colors.
# This avoids the Matplotlib ValueError caused by mixing
# per-point alpha arrays and scalar set_alpha().
# ------------------------------------------------------------

cluster_colors = np.ones(
    (N_STARS, 4),
    dtype=float
)

# White RGB
cluster_colors[:, 0] = 1.0
cluster_colors[:, 1] = 1.0
cluster_colors[:, 2] = 1.0

# Individual alpha
cluster_colors[:, 3] = alphas


cluster_scatter = ax.scatter(
    xs,
    ys,
    s=sizes,
    c=cluster_colors,
    linewidths=0,
    zorder=2
)


# Immutable copy for animation
cluster_base_alphas = alphas.copy()


# ============================================================
# BACKGROUND STAR FIELD
# ============================================================

N_BACKGROUND = 150

bx = rng.uniform(
    -1.2,
    1.2,
    N_BACKGROUND
)

by = rng.uniform(
    -1.0,
    1.0,
    N_BACKGROUND
)

background_sizes = rng.uniform(
    0.3,
    2.5,
    N_BACKGROUND
)

background_alphas = rng.uniform(
    0.04,
    0.18,
    N_BACKGROUND
)


background_colors = np.ones(
    (N_BACKGROUND, 4)
)

background_colors[:, 3] = background_alphas


ax.scatter(
    bx,
    by,
    s=background_sizes,
    c=background_colors,
    linewidths=0,
    zorder=0
)


# ============================================================
# CLUSTER BOUNDARY
# ============================================================

cluster_radius = Circle(
    (0, 0),
    0.78,
    fill=False,
    edgecolor="#47718f",
    linewidth=1,
    linestyle=(0, (4, 6)),
    alpha=0.32
)

ax.add_patch(cluster_radius)


ax.text(
    -1.10,
    -0.91,
    "STAR CLUSTER",
    color="#607e98",
    fontsize=8
)


# ============================================================
# PLANETARY NEBULA
# ============================================================

pn_x = 0.22
pn_y = 0.08


# Several rings produce a cheap glow
pn_glows = []

pn_shell_settings = [
    (0.115, 22, 0.025),
    (0.105, 12, 0.045),
    (0.095,  6, 0.100),
    (0.085,  2, 0.450),
]


for radius, lw, alpha in pn_shell_settings:

    c = Circle(
        (pn_x, pn_y),
        radius,
        fill=False,
        edgecolor="#65ddff",
        linewidth=lw,
        alpha=alpha,
        zorder=4
    )

    ax.add_patch(c)
    pn_glows.append(c)


# Inner shell
pn_inner = Circle(
    (pn_x, pn_y),
    0.052,
    fill=False,
    edgecolor="#b58aff",
    linewidth=2,
    alpha=0.55,
    zorder=5
)

ax.add_patch(pn_inner)


# Central-star glow
pn_star_glow = ax.scatter(
    [pn_x],
    [pn_y],
    s=170,
    c="white",
    alpha=0.13,
    linewidths=0,
    zorder=7
)


# Central star
pn_star = ax.scatter(
    [pn_x],
    [pn_y],
    s=18,
    c="white",
    alpha=1.0,
    linewidths=0,
    zorder=8
)


# PN label
ax.text(
    pn_x + 0.12,
    pn_y + 0.13,
    "PN",
    color="#8de6ff",
    fontsize=8,
    weight="bold"
)


# ============================================================
# PROPER MOTION ARROWS
# ============================================================

# Several representative cluster stars
arrow_stars = np.array([
    [-0.45,  0.30],
    [-0.30, -0.28],
    [ 0.05,  0.38],
    [ 0.42, -0.22],
    [ 0.53,  0.25],
])


# Schematic common proper motion
motion_dx = 0.10
motion_dy = 0.045


motion_arrows = []


for x, y in arrow_stars:

    q = ax.arrow(
        x,
        y,
        motion_dx,
        motion_dy,
        width=0.003,
        head_width=0.025,
        head_length=0.035,
        length_includes_head=True,
        color="#5bc6ff",
        alpha=0,
        zorder=6
    )

    motion_arrows.append(q)


# PN proper-motion arrow
pn_arrow = ax.arrow(
    pn_x,
    pn_y,
    motion_dx,
    motion_dy,
    width=0.004,
    head_width=0.030,
    head_length=0.040,
    length_includes_head=True,
    color="#ffffff",
    alpha=0,
    zorder=9
)


# ============================================================
# RIGHT PANEL BACKGROUND
# ============================================================

box = FancyBboxPatch(
    (0.02, 0.02),
    0.96,
    0.96,
    boxstyle="round,pad=0.015,rounding_size=0.02",
    edgecolor="#24384c",
    facecolor="#050914",
    linewidth=1
)

panel.add_patch(box)


panel.text(
    0.08,
    0.93,
    "ASSOCIATION TEST",
    color="#8da5bb",
    fontsize=9,
    weight="bold"
)


# ============================================================
# SELECTION TESTS
# ============================================================

tests = [
    (
        "POSITION",
        "277 positional pairs"
    ),
    (
        "PARALLAX",
        "Gaia consistency"
    ),
    (
        "PROPER MOTION",
        "38 kinematic candidates"
    ),
    (
        "EXTINCTION",
        "15 compatible"
    ),
    (
        "STELLAR EVOLUTION",
        "7 higher-likelihood"
    ),
]


test_y = [
    0.78,
    0.64,
    0.50,
    0.36,
    0.22
]


test_labels = []
test_values = []
test_dots = []


# Vertical timeline
panel.plot(
    [0.10, 0.10],
    [0.235, 0.765],
    color="#24384c",
    lw=1,
    zorder=0
)


for (label, value), y in zip(
    tests,
    test_y
):

    dot = Circle(
        (0.10, y),
        0.015,
        facecolor="#304355",
        edgecolor="none",
        zorder=3
    )

    panel.add_patch(dot)

    test_dots.append(dot)


    t1 = panel.text(
        0.15,
        y + 0.014,
        label,
        color="#60788e",
        fontsize=9,
        va="center",
        weight="bold"
    )


    t2 = panel.text(
        0.15,
        y - 0.030,
        value,
        color="#465d70",
        fontsize=7.5,
        va="center"
    )


    test_labels.append(t1)
    test_values.append(t2)


# ============================================================
# STATUS
# ============================================================

status_text = panel.text(
    0.08,
    0.08,
    "",
    color="white",
    fontsize=10,
    weight="bold"
)


# ============================================================
# LARGE COUNTER
# ============================================================

counter = ax.text(
    -1.10,
    0.88,
    "",
    color="white",
    fontsize=24,
    weight="bold"
)


subcounter = ax.text(
    -1.10,
    0.79,
    "",
    color="#718ba1",
    fontsize=8
)


# ============================================================
# HELPERS
# ============================================================

def smoothstep(x):

    x = np.clip(
        x,
        0.0,
        1.0
    )

    return x * x * (3.0 - 2.0 * x)


def set_test_state(
    index,
    strength
):

    strength = float(
        np.clip(
            strength,
            0.0,
            1.0
        )
    )

    # Inactive -> bright cyan/white
    dot_color = (
        0.20 + 0.20 * strength,
        0.30 + 0.55 * strength,
        0.38 + 0.55 * strength
    )

    label_color = (
        0.38 + 0.50 * strength,
        0.47 + 0.45 * strength,
        0.56 + 0.40 * strength
    )

    value_color = (
        0.28 + 0.38 * strength,
        0.36 + 0.52 * strength,
        0.44 + 0.50 * strength
    )


    test_dots[index].set_facecolor(
        dot_color
    )

    test_labels[index].set_color(
        label_color
    )

    test_values[index].set_color(
        value_color
    )


# ============================================================
# TIMELINE
# ============================================================

# Normalized start points
starts = np.array([
    0.07,   # position
    0.24,   # parallax
    0.39,   # proper motion
    0.57,   # extinction
    0.74    # stellar evolution
])

test_duration = 0.10


# ============================================================
# ANIMATION UPDATE
# ============================================================

def update(frame):

    f = frame / (NFRAMES - 1)


    # --------------------------------------------------------
    # PANEL PROGRESSION
    # --------------------------------------------------------

    for i, start in enumerate(starts):

        strength = smoothstep(
            (f - start) / test_duration
        )

        set_test_state(
            i,
            strength
        )


    # --------------------------------------------------------
    # STAGE TEXT
    # --------------------------------------------------------

    if f < starts[0]:

        status = "SEARCHING..."

        count = ""

        subtitle = (
            "Cross-matching planetary nebulae and clusters"
        )


    elif f < starts[1]:

        status = "POSITIONAL MATCH"

        count = "277"

        subtitle = (
            "potential positional pairs"
        )


    elif f < starts[2]:

        status = "CHECKING GAIA"

        count = "277"

        subtitle = (
            "testing distance and motion"
        )


    elif f < starts[3]:

        status = "KINEMATICALLY CONSISTENT"

        count = "38"

        subtitle = (
            "candidate PN–cluster pairs"
        )


    elif f < starts[4]:

        status = "EXTINCTION COMPATIBLE"

        count = "15"

        subtitle = (
            "candidates after extinction comparison"
        )


    elif f < 0.91:

        status = "EVOLUTION COMPATIBLE"

        count = "7"

        subtitle = (
            "higher-likelihood associations"
        )


    else:

        status = "FOLLOW-UP REQUIRED"

        count = "7"

        subtitle = (
            "radial velocities and additional data"
        )


    status_text.set_text(
        status
    )

    counter.set_text(
        count
    )

    subcounter.set_text(
        subtitle
    )


    # --------------------------------------------------------
    # PROPER-MOTION ARROWS
    # --------------------------------------------------------

    arrow_fade_in = smoothstep(
        (f - 0.29) / 0.07
    )

    arrow_fade_out = (
        1.0
        -
        smoothstep(
            (f - 0.58) / 0.08
        )
    )

    arrow_alpha = (
        arrow_fade_in
        *
        arrow_fade_out
    )


    for arrow in motion_arrows:

        arrow.set_alpha(
            0.65 * arrow_alpha
        )


    pn_arrow.set_alpha(
        0.95 * arrow_alpha
    )


    # --------------------------------------------------------
    # PLANETARY NEBULA PULSE
    # --------------------------------------------------------

    pulse = (
        0.5
        +
        0.5 * np.sin(
            frame * 0.09
        )
    )


    base_shell_alphas = [
        0.025,
        0.045,
        0.100,
        0.450
    ]


    for shell, base_alpha in zip(
        pn_glows,
        base_shell_alphas
    ):

        shell.set_alpha(
            base_alpha
            *
            (
                0.82
                +
                0.18 * pulse
            )
        )


    pn_inner.set_alpha(
        0.45
        +
        0.10 * pulse
    )


    pn_star_glow.set_alpha(
        0.09
        +
        0.08 * pulse
    )


    # --------------------------------------------------------
    # DIM CLUSTER DURING FINAL SELECTION
    # --------------------------------------------------------
    #
    # FIXED VERSION:
    # We change the alpha channel of each RGBA color instead
    # of calling cluster_scatter.set_alpha(scalar).
    #
    # This preserves individual stellar brightness and avoids
    # the "truth value of an array is ambiguous" error.
    # --------------------------------------------------------

    dim = smoothstep(
        (f - 0.70) / 0.18
    )


    fade_factor = (
        1.0
        -
        0.55 * dim
    )


    current_colors = cluster_colors.copy()

    current_colors[:, 3] = (
        cluster_base_alphas
        *
        fade_factor
    )


    cluster_scatter.set_facecolors(
        current_colors
    )


    # --------------------------------------------------------
    # Slight emphasis on PN during final stages
    # --------------------------------------------------------

    emphasis = smoothstep(
        (f - 0.68) / 0.18
    )


    pn_star.set_sizes([
        18
        +
        10 * emphasis
    ])


    # blit=False, so returning artists is unnecessary
    return []


# ============================================================
# CREATE ANIMATION
# ============================================================

anim = FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    interval=1000 / FPS,
    blit=False,
    repeat=True
)


# ============================================================
# EXPORT
# ============================================================

print(
    f"Rendering {NFRAMES} frames "
    f"({DURATION} s at {FPS} fps)"
)

print(
    f"Output: {OUTFILE}"
)


if OUTPUT.lower() == "webm":

    writer = FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2200,
        extra_args=[
            "-pix_fmt",
            "yuv420p",

            "-row-mt",
            "1",

            "-deadline",
            "good",

            "-cpu-used",
            "2"
        ]
    )


else:

    writer = FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=3500,
        extra_args=[
            "-pix_fmt",
            "yuv420p",

            "-preset",
            "medium",

            "-movflags",
            "+faststart"
        ]
    )


anim.save(
    OUTFILE,
    writer=writer,
    dpi=DPI
)


plt.close(fig)


# ============================================================
# RESULT
# ============================================================

size_mb = (
    os.path.getsize(OUTFILE)
    /
    (1024 * 1024)
)


print()
print("Done.")
print(
    f"File: {OUTFILE}"
)
print(
    f"Size: {size_mb:.1f} MB"
)


display(
    Video(
        OUTFILE,
        embed=True,
        html_attributes=(
            "controls loop autoplay muted"
        )
    )
)

# Как найти связь между планетарной туманностью и звёздным скоплением

Случайно увидеть планетарную туманность в направлении звёздного скопления недостаточно, чтобы считать их физически связанными. Исследование начинается с поиска позиционных совпадений: первоначальный кросс-матч дал 277 потенциальных пар «центральная звезда планетарной туманности — звёздное скопление». Затем кандидаты последовательно проверяются с помощью астрометрических данных Gaia — по параллаксу и собственному движению. После этих проверок авторы выделили 38 пар с совместимыми кинематическими характеристиками.  

Следующий этап — сравнение межзвёздного поглощения и физических характеристик центральных звёзд. Для 15 кандидатов значения экстинкции оказались совместимыми, после чего положение центральных звёзд на диаграмме температура–светимость сравнивалось с моделями пост-AGB эволюции. В результате семь систем были выделены как кандидаты с более высокой вероятностью физической связи. Однако они всё ещё требуют независимого подтверждения — в частности, дополнительных измерений радиальных скоростей и других данных.  

Анимация схематически показывает процедуру отбора кандидатов и не изображает пространственное расположение конкретного звёздного скопления.


# How to Identify a Planetary Nebula–Star Cluster Association

Simply finding a planetary nebula in the direction of a star cluster is not sufficient to establish a physical association. The study begins by searching for positional coincidences: the initial cross-match produced 277 potential central-star–cluster pairs. The candidates are then tested using Gaia astrometry, including parallax and proper motion. After these checks, the authors identified 38 pairs with compatible kinematic properties.  

The next stage compares interstellar extinction and the physical properties of the planetary-nebula central stars. Fifteen candidates were found to have compatible extinction values, after which the locations of their central stars on the temperature–luminosity diagram were compared with post-AGB evolutionary models. Seven systems emerged as higher-likelihood physical associations. They still require independent confirmation, however, particularly through additional radial-velocity measurements and other follow-up data.  

The animation provides a schematic representation of the candidate-selection procedure and does not depict the actual spatial distribution of any particular star cluster.